In [ ]:
import kagglehub

path = kagglehub.dataset_download("lantian773030/pokemonclassification")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, models, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt
import os
import time


device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
data_dir = os.path.join(path, "PokemonData")
num_classes = 150 
batch_size = 32


transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(), 
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size
train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

dataloaders = {
    'train': DataLoader(train_dataset, batch_size=batch_size, shuffle=True),
    'val': DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
}
dataset_sizes = {'train': len(train_dataset), 'val': len(val_dataset)}

In [ ]:
def get_model(model_name):
    if model_name == "alexnet":
        model = models.alexnet(weights=models.AlexNet_Weights.DEFAULT)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
    elif model_name == "googlenet":
        model = models.googlenet(weights=models.GoogLeNet_Weights.DEFAULT)
        model.aux_logits = False 
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "resnet18":
        model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "resnet34":
        model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
    elif model_name == "efficientnet":
        model = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.DEFAULT)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    
    return model.to(device)

In [ ]:
def train_model(model, criterion, optimizer, num_epochs=5):
    history = {'train_acc': [], 'val_acc': [], 'train_loss': [], 'val_loss': []}
    since = time.time()

    for epoch in range(num_epochs):
        for phase in ['train', 'val']:
            if phase == 'train': model.train()
            else: model.eval()

            running_loss, running_corrects = 0.0, 0

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    outputs = model(inputs)
                    _, preds = torch.max(outputs, 1)
                    loss = criterion(outputs, labels)
                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)

            epoch_loss = running_loss / dataset_sizes[phase]
            epoch_acc = running_corrects.double() / dataset_sizes[phase]

            if phase == 'train':
                history['train_acc'].append(epoch_acc.item())
                history['train_loss'].append(epoch_loss)
            else:
                history['val_acc'].append(epoch_acc.item())
                history['val_loss'].append(epoch_loss)

        print(f'Epoch {epoch}/{num_epochs - 1} | '
              f'Train Acc: {history["train_acc"][-1]:.4f} | '
              f'Val Acc: {history["val_acc"][-1]:.4f}')

    time_elapsed = time.time() - since
    print(f'학습 완료, 소요 시간: {time_elapsed // 60:.0f}m {time_elapsed % 60:.0f}s')
    return model, history

In [ ]:
experiments = ["alexnet", "googlenet", "resnet18", "resnet34", "efficientnet"]
all_results = {}

for name in experiments:
    print(f"\n학습: {name}")
    model = get_model(name)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=0.001) 
    
    best_model, history = train_model(model, criterion, optimizer, num_epochs=5)
    all_results[name] = history
    
   
    torch.save(best_model.state_dict(), f"pokemon_{name}.pth")

In [ ]:
import matplotlib.pyplot as plt

def plot_history(all_results):
    if not all_results:
        print("시각화할 데이터가 없습니다. 학습이 완료되었는지 확인하세요.")
        return

    plt.figure(figsize=(15, 5))
    

    plt.subplot(1, 2, 1)
    for name, history in all_results.items():
        if 'val_acc' in history and 'train_acc' in history:
            line, = plt.plot(history['val_acc'], label=f'{name} (Val)', marker='o')
            plt.plot(history['train_acc'], label=f'{name} (Train)', 
                     linestyle='--', alpha=0.6, color=line.get_color())
        
    plt.title('Accuracy Comparison (Train vs Val)')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy')
    plt.legend()
    plt.grid(True, alpha=0.3)


    plt.subplot(1, 2, 2)
    for name, history in all_results.items():
        if 'val_loss' in history and 'train_loss' in history:
            line, = plt.plot(history['val_loss'], label=f'{name} (Val)', marker='o')
            plt.plot(history['train_loss'], label=f'{name} (Train)', 
                     linestyle='--', alpha=0.6, color=line.get_color())
        
    plt.title('Loss Comparison (Train vs Val)')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.savefig('learning_curves.png') 
    plt.show() 


plot_history(all_results)